# 00. OT & Agentic Analytics 전체 데모 — 완성형 SQL 에이전트
> Day 1 · 1H · 소요 약 50분

## 이 노트북의 역할

**이 노트북은 "구경만" 하세요.** Day 1 첫 시간에 강사가 실행을 보여주고, 여러분은 4일 뒤 직접 만들 최종 결과물을 미리 체험합니다.

- **왜 맨 처음에 보여주는가?** 재료부터 쌓기 전에 완성품을 먼저 맛보게 하는 **역방향 학습(reverse learning)** 때문입니다. 4일간의 이정표가 됩니다.
- **지금 이해 못 해도 OK.** `TypedDict`·`StateGraph`·프롬프트 엔지니어링·가드레일 등 모든 개념을 Day 1~3에 차근차근 배웁니다.
- **Day 3 20H** 에 17번 노트북 `17_my_sql_agent.ipynb` 에서 **본인 손으로 같은 구조** 를 만듭니다. 이 노트북은 바로 그 17번의 압축판입니다.

## 오늘 최종 결과물 미리보기

```
사용자 질문 ──▶ [생성 SQL] ──▶ [실행] ──▶ [검증]
                                          │
           ┌──────── retry (최대 3회) ◀───┤ error?
           │                              │
           ▼                              ▼
      [SQL 재생성]              [자연어 답변 생성]
```

4개 노드가 **재시도 루프** 로 연결된 LangGraph 에이전트. Day 3 19~20H 에서 같은 그래프를 직접 짭니다.

In [ ]:
%pip install -q langgraph langchain langchain-openai sqlalchemy psycopg2-binary sqlparse pandas tabulate

In [ ]:
# 이 셀은 "API 키 / DB 접속 정보 같은 비밀(Secret) 을 안전하게 읽어 오는" 도우미입니다.
# Colab 의 [🔑 Secrets] 탭에 넣어 둔 값이 있으면 그걸 읽고, 없으면 입력창을 띄워 받습니다.
# 코드에 키를 직접 적어 두면 노트북을 공유할 때 노출될 위험이 있어 매 노트북 첫 셀에서 이렇게 처리합니다.
import os  # os.environ 으로 운영체제의 환경변수에 접근


def _load_secret(key: str, required: bool = True) -> None:
    """환경변수 `key` 를 채워 넣는다. Colab → getpass(터미널 입력) 순서로 시도."""
    # 1) 이미 환경변수에 들어 있으면 그대로 둔다 (재실행 시 키 재입력 방지).
    if os.environ.get(key):
        return

    value = None
    # 2) Colab 환경이면 [🔑 Secrets] 패널에서 읽어 본다.
    try:
        from google.colab import userdata  # type: ignore  (Colab 전용 모듈)
        value = userdata.get(key)
    except Exception:
        # 로컬 PC 등 Colab 이 아닌 환경에서는 google.colab import 자체가 실패 → 무시.
        value = None

    # 3) 그래도 못 찾으면 입력창(비밀번호 형식)으로 직접 입력받는다.
    if not value:
        try:
            from getpass import getpass  # 입력값을 화면에 안 보여주는 안전한 input
            value = getpass(f"Enter {key}: ")
        except Exception:
            value = None

    # 4) 최종 처리: 값이 있으면 환경변수에 저장, 없는데 필수면 명시적 에러.
    if value:
        os.environ[key] = value
    elif required:
        raise RuntimeError(f"{key} is not set. Register it in Colab Secrets or via env var.")


# 이 노트북에서 꼭 필요한 두 키를 환경변수에 적재한다.
_load_secret("OPENAI_API_KEY", required=True)  # OpenAI 호출용
_load_secret("NEON_DSN", required=True)         # Neon PostgreSQL 접속 문자열
print("Environment ready.")

## 데모 에이전트 조립 — 한 덩어리

원래 17번 노트북에서는 상태 / 가드레일 / 4노드 / 그래프 조립을 단계별 **40여 셀**로 나눠 설명하지만, 이 데모에서는 한 번에 정의합니다. 각 부분의 의미는 아래 주석으로만 가볍게 표시했습니다 — **상세 학습은 Day 3 19~20H.**

> **전제**: 병원 DB (`patients`, `doctors`, `visits`, `diagnoses`, `departments`) 가 Neon 에 적재되어 있다고 가정합니다. 없으면 `01_postgres_basics.ipynb` 를 먼저 실행 후 돌아오세요. 테이블이 하나도 없으면 코드가 현재 DB의 아무 테이블이나 잡아 스키마를 채우지만, 이 데모의 질문들은 병원 DB 를 전제로 작성되어 있습니다.

In [ ]:
# =====================================================================
# 데모 에이전트 전체 조립 (한 셀)
# =====================================================================
# 이 한 셀에서 다음 다섯 가지를 한꺼번에 정의합니다. 17번 노트북에서는 같은 내용을
# 약 40개 셀로 쪼개서 차근차근 배웁니다 — 오늘은 "이런 모습이구나" 정도만 확인하세요.
#   (1) 라이브러리 임포트 + DB 엔진 생성
#   (2) 스키마 텍스트 추출 함수
#   (3) 에이전트의 "공유 메모"인 AgentState (LangGraph 의 핵심 개념)
#   (4) SQL 안전 가드 + LLM 클라이언트 + 프롬프트 템플릿
#   (5) 4개 노드(generate_sql / execute_sql / validate_sql / generate_answer) + 그래프 조립
# =====================================================================

# --- (1-a) 표준 / 외부 라이브러리 ---
import re  # 정규식: SQL 가드와 코드펜스 제거에 사용
from typing import TypedDict, Optional, List, Dict, Any  # 상태 타입 명세 (정적 힌트)

from sqlalchemy import create_engine, inspect, text
#   create_engine: DB 접속 객체(엔진) 생성
#   inspect      : 테이블/컬럼/FK 같은 스키마 메타데이터를 읽는 헬퍼
#   text         : 원시 SQL 문자열을 SQLAlchemy 가 안전하게 처리하도록 감싸는 함수
import pandas as pd  # 표 형태 결과를 DataFrame 으로 다루기 위해

# LangGraph: "노드(함수) + 엣지(연결) + 상태" 로 에이전트 흐름을 그래프로 그리는 라이브러리
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI            # OpenAI 챗 모델 래퍼
from langchain_core.prompts import ChatPromptTemplate  # 변수 치환이 가능한 프롬프트 템플릿
from langchain_core.output_parsers import StrOutputParser  # LLM 응답 → 문자열로 정규화


# --- (1-b) DB 엔진 (가능하면 "읽기 전용" 모드로 시도) ---
# 의미: LLM 이 실수로 DROP/DELETE 같은 변경 SQL 을 만들어도, 세션 자체가 read-only 라면
# DB 에서 거부됩니다. 정규식 가드는 보조 방어선, 진짜 방어선은 DB 권한입니다.
try:
    engine = create_engine(
        os.environ["NEON_DSN"],
        # PostgreSQL 세션 옵션: 트랜잭션을 read-only 로 강제
        connect_args={"options": "-c default_transaction_read_only=on"},
        pool_pre_ping=True,  # 끊긴 커넥션을 자동 재시도 (Neon 같은 서버리스에서 권장)
    )
    # 실제 연결 테스트 — `SELECT 1` 은 "DB가 살아 있는지" 확인하는 관용구
    with engine.connect() as conn:
        conn.execute(text("SELECT 1"))
    print("Engine connected in read-only mode.")
except Exception as e:
    # 일부 호스팅 환경에서는 read-only 옵션이 막혀 있어 실패할 수 있음 → 일반 모드로 폴백
    print(f"[WARN] read-only 옵션 실패 → 일반 모드로 재연결: {e}")
    engine = create_engine(os.environ["NEON_DSN"], pool_pre_ping=True)
    with engine.connect() as conn:
        conn.execute(text("SELECT 1"))
    print("Engine connected (non-RO).")


# --- (2) 스키마 텍스트 추출 함수 ---
# LLM 은 DB 에 직접 접속하지 못하므로, 우리가 "프롬프트 안에 스키마를 글자로 적어"
# 알려 줘야 합니다. 이 함수는 CREATE TABLE 형태 + COMMENT 라인으로 변환합니다.
def collect_schema(engine, tables=None) -> str:
    """DB 스키마를 LLM 프롬프트용 DDL 텍스트로 변환 (NB17과 동일 로직)."""
    inspector = inspect(engine)
    if tables is None:
        # 별도 지정이 없으면 public 스키마의 모든 테이블을 가져온다.
        tables = inspector.get_table_names()

    parts = []
    for table in tables:
        # 컬럼 메타데이터: [{name, type, nullable, ...}, ...]
        columns = inspector.get_columns(table)
        # 외래키 메타데이터: [{constrained_columns, referred_table, referred_columns, ...}]
        fks = inspector.get_foreign_keys(table)

        # 컬럼 한 줄씩 "    이름 타입 [NOT NULL]" 로 직렬화
        col_lines = []
        for col in columns:
            nullable = "" if col["nullable"] else " NOT NULL"
            col_lines.append(f"    {col['name']} {col['type']}{nullable}")

        # 외래키도 표준 DDL 문법대로 직렬화
        fk_lines = []
        for fk in fks:
            fk_lines.append(
                f"    FOREIGN KEY ({', '.join(fk['constrained_columns'])}) "
                f"REFERENCES {fk['referred_table']}({', '.join(fk['referred_columns'])})"
            )

        # CREATE TABLE 블록 조립 (사람이 읽기 좋고 LLM 도 익숙한 형식)
        ddl = f"CREATE TABLE {table} (\n"
        ddl += ",\n".join(col_lines)
        if fk_lines:
            ddl += ",\n" + ",\n".join(fk_lines)
        ddl += "\n);"

        # PostgreSQL 의 COMMENT ON COLUMN 을 함께 붙여 LLM 이 의미를 추측하지 않게 한다.
        # (예: gender 컬럼에 'M=남성, F=여성' 같은 한국어 설명)
        try:
            with engine.connect() as conn:
                comments = conn.execute(
                    text("""
                        SELECT a.attname,
                               col_description(c.oid, a.attnum) AS comment
                        FROM pg_class c
                        JOIN pg_namespace n ON n.oid = c.relnamespace
                        JOIN pg_attribute a ON a.attrelid = c.oid
                        WHERE c.relname = :table
                          AND n.nspname = 'public'
                          AND a.attnum > 0
                          AND NOT a.attisdropped
                        ORDER BY a.attnum
                    """),
                    {"table": table},  # :table 자리표시자에 안전하게 바인딩
                ).fetchall()
            for col_name, comment in comments:
                if comment:
                    # `-- 테이블.컬럼: 설명` 한 줄을 DDL 뒤에 부록으로 추가
                    ddl += f"\n-- {table}.{col_name}: {comment}"
        except Exception:
            # 커멘트 조회는 부가 기능이므로 실패해도 데모는 계속.
            pass

        parts.append(ddl)
    # 테이블별 DDL 블록을 빈 줄로 연결한 거대한 문자열 한 덩어리로 반환
    return "\n\n".join(parts)


# 이 데모는 병원 DB 를 가정하지만, 실제 DB 에 그 테이블이 없으면 자동 폴백한다.
TABLES = ["patients", "doctors", "visits", "diagnoses", "departments"]
_available = set(inspect(engine).get_table_names())
TABLES = [t for t in TABLES if t in _available]
if not TABLES:
    TABLES = list(_available)[:10]  # 어떤 테이블도 없으면 임의의 상위 10개로
print(f"Tables to include in schema: {TABLES}")

SCHEMA = collect_schema(engine, TABLES)  # 위에서 만든 함수로 거대한 스키마 문자열 생성
print(f"\nSchema text length: {len(SCHEMA)} chars")


# --- (3) AgentState — LangGraph 의 "공유 칠판" ---
# 그래프 안의 모든 노드(함수)는 이 dict 를 입력으로 받고, 일부 키를 채워 반환합니다.
# 반환된 부분은 LangGraph 가 자동으로 기존 state 에 합쳐 다음 노드로 넘깁니다.
# total=False  → 모든 키를 항상 채울 필요는 없다는 뜻.
class AgentState(TypedDict, total=False):
    question: str               # 사용자가 입력한 자연어 질문
    sql: str                    # LLM 이 생성한 SQL
    result: List[Dict[str, Any]]  # 실행 결과 (행 리스트)
    result_md: str              # 결과를 Markdown 표로 직렬화한 문자열 (LLM 답변용)
    answer: str                 # 최종 자연어 답변
    error: Optional[str]        # 직전 단계 에러 메시지 (재시도 트리거)
    retry_count: int            # 재시도 횟수 (3회 도달 시 포기)


# --- (4-a) SQL 안전 가드 (정규식 1차 방어선) ---
# 영문 키워드만 검사하므로 진짜 보안은 위의 read-only DB 세션이 담당합니다.
BLOCKED_SQL_PATTERN = re.compile(
    r"\b(DROP|DELETE|UPDATE|INSERT|ALTER|TRUNCATE|CREATE|GRANT|REVOKE)\b",
    re.IGNORECASE,  # 대문자/소문자 무관
)


def is_safe_sql(sql: str) -> tuple[bool, str]:
    """위험 키워드가 보이면 (False, 사유) 를 반환."""
    match = BLOCKED_SQL_PATTERN.search(sql)
    if match:
        return False, f"보안 위반: '{match.group()}' 명령은 허용되지 않습니다."
    return True, ""


def inject_limit(sql: str, cap: int = 1000) -> str:
    """LIMIT 절이 없으면 자동으로 LIMIT 1000 을 덧붙여 대량 결과를 막는다."""
    stripped = sql.strip().rstrip(";")
    # 이미 LIMIT 가 있으면 그대로 둔다 (사용자/LLM 이 지정한 값 우선).
    if re.search(r"\bLIMIT\s+\d+\b", stripped, re.IGNORECASE):
        return stripped
    return f"{stripped}\nLIMIT {cap}"


def strip_sql_fences(sql: str) -> str:
    """LLM 이 ```sql … ``` 같은 코드펜스를 같이 보내면 제거한다."""
    sql = re.sub(r"```sql\s*", "", sql, flags=re.IGNORECASE)
    sql = re.sub(r"```\s*", "", sql)
    return sql.strip()


# --- (4-b) LLM 두 개: SQL 생성용(엄격) / 답변 생성용(자연스럽게) ---
# temperature 는 "창의성" 손잡이. SQL 은 0(매번 같은 답) 이 안전하고,
# 자연어 답변은 0.2 정도로 살짝 풀어 주면 어색함이 줄어듭니다.
sql_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
answer_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)


# --- (4-c) SQL 생성 프롬프트 ---
# {중괄호} 부분이 .invoke() 호출 시 실제 값으로 치환됩니다.
# 규칙을 명시적으로 적어 둘수록 LLM 의 잘못된 추측이 줄어듭니다.
SQL_GEN_TEMPLATE = ChatPromptTemplate.from_template(
    """당신은 PostgreSQL 전문가입니다. 아래 스키마를 참고하여 질문에 대한 SQL 하나를 작성하세요.

## 스키마
{schema}

## 규칙
- SELECT 문만 작성. DML/DDL 금지.
- visits.status = 'completed' 만 유효한 진료로 간주.
- 나이 = EXTRACT(YEAR FROM AGE(birth_date))
- 결과가 많을 가능성이 있으면 LIMIT 100 이하를 권장.
- 설명 없이 **SQL 만** 반환.
{error_feedback}

## 질문
{question}

SQL:
"""
)

# LCEL 파이프 연산: (프롬프트) → (LLM) → (문자열 파서). Day 3 15H 에서 자세히 다룹니다.
sql_gen_chain = SQL_GEN_TEMPLATE | sql_llm | StrOutputParser()


# --- (5-a) 노드 1: SQL 생성 ---
# LangGraph 노드 함수의 약속:
#   - 입력: state (AgentState dict)
#   - 출력: dict (state 에 덮어쓸 키들)
def generate_sql(state: AgentState) -> dict:
    """노드 1 — 질문 → SQL. 이전 에러가 있으면 피드백으로 재생성."""
    # 직전 시도가 실패했다면, 그 에러를 프롬프트에 끼워 LLM 이 같은 실수를 반복하지 않게 한다.
    error_feedback = ""
    if state.get("error"):
        error_feedback = (
            "\n## 직전 시도의 실패\n"
            f"- 실패 SQL:\n{state.get('sql', '')}\n"
            f"- 에러: {state['error']}\n"
            "이 에러를 피해서 SQL 을 다시 작성하세요."
        )

    # 프롬프트 → LLM → 문자열 한 번에 흐르게 한 LCEL 체인을 실제로 호출
    raw = sql_gen_chain.invoke({
        "schema": SCHEMA,
        "question": state["question"],
        "error_feedback": error_feedback,
    })
    sql = strip_sql_fences(raw)  # 코드펜스 제거
    return {
        "sql": sql,
        # state 의 retry_count 는 시도 횟수. dict.get(..., 0) 으로 None 안전 처리.
        "retry_count": state.get("retry_count", 0) + 1,
    }


# --- (5-b) 노드 2: SQL 실행 ---
def execute_sql(state: AgentState) -> dict:
    """노드 2 — SQL 실행 + 결과 Markdown 화."""
    sql = state.get("sql", "")
    if not sql:
        return {"error": "실행할 SQL 이 없습니다.", "result": [], "result_md": ""}

    # 1차 정규식 가드 (영문 DML/DDL 키워드 차단)
    ok, reason = is_safe_sql(sql)
    if not ok:
        return {"error": reason, "result": [], "result_md": ""}

    # LIMIT 자동 주입으로 의도치 않은 대량 결과를 차단
    safe_sql = inject_limit(sql, cap=1000)
    try:
        # text() 로 감싸서 SQLAlchemy 가 % 같은 특수문자를 자동 이스케이프하게 한다.
        # 이 처리를 빼면 LIKE '%어쩌구%' 같은 SQL 에서 psycopg2 가 오작동합니다 (01번 이슈).
        df = pd.read_sql(text(safe_sql), engine)
        if df.empty:
            return {
                "result": [],
                "result_md": "(결과 없음 — 조건을 다시 확인하세요)",
                "error": "",
                "sql": safe_sql,
            }
        # 결과는 두 형태로 보관: (1) LLM 답변용 Markdown 표, (2) 디버깅용 dict 리스트.
        # 50행으로 잘라 LLM 컨텍스트를 아끼고 토큰 비용을 통제한다.
        result_rows = df.head(50).to_dict(orient="records")
        md_table = df.head(50).to_markdown(index=False)
        if len(df) > 50:
            md_table += f"\n\n... 외 {len(df) - 50}행"
        return {
            "result": result_rows,
            "result_md": md_table,
            "error": "",
            "sql": safe_sql,
        }
    except Exception as e:
        # 에러 메시지는 길이를 줄여 다음 노드의 프롬프트에 그대로 주입할 수 있게 한다.
        return {
            "error": f"SQL 실행 오류: {type(e).__name__}: {str(e)[:300]}",
            "result": [],
            "result_md": "",
        }


# --- (5-c) 노드 3: 검증 ---
# 이 데모에서는 단순히 통과시킵니다. 실전에서는 EXPLAIN, 행수 sanity check 등을 추가합니다.
def validate_sql(state: AgentState) -> dict:
    """노드 3 — 단순 검증. 에러가 있으면 그대로 두고 분기 함수가 retry 결정."""
    if state.get("error"):
        return {}  # 아무것도 바꾸지 않으면 LangGraph 는 기존 state 유지
    return {"error": ""}


# --- (5-d) 답변 생성 프롬프트 ---
ANSWER_TEMPLATE = ChatPromptTemplate.from_template(
    """다음 SQL 실행 결과를 바탕으로 질문에 한국어로 답변하세요.

## 질문
{question}

## 실행한 SQL
{sql}

## 결과 (Markdown 표)
{result_md}

## 규칙
- 2-3 문장으로 핵심만 요약.
- 숫자에 천 단위 구분자(쉼표) 사용.
- 결과가 비어 있으면 "해당 조건에 맞는 데이터가 없습니다." 로 시작하는 안내.
- 추측 금지 — 표에 없는 수치는 언급하지 말 것.
"""
)

answer_chain = ANSWER_TEMPLATE | answer_llm | StrOutputParser()


# --- (5-e) 노드 4: 자연어 답변 ---
def generate_answer(state: AgentState) -> dict:
    """노드 4 — 결과 → 자연어 답변."""
    # 3회 재시도 후에도 에러가 남아 있으면 깨끗하게 포기 메시지를 돌려준다.
    if state.get("error") and state.get("retry_count", 0) >= 3:
        return {
            "answer": (
                "죄송합니다. 질문에 답변하지 못했습니다.\n"
                f"- 마지막 에러: {state['error']}\n"
                "- 질문을 더 구체적으로 다시 물어봐 주세요."
            )
        }
    ans = answer_chain.invoke({
        "question": state.get("question", ""),
        "sql": state.get("sql", ""),
        # 결과 표가 너무 길면 1500자에서 자르기 (토큰 절약)
        "result_md": (state.get("result_md") or "(결과 없음)")[:1500],
    })
    return {"answer": ans.strip()}


# --- (5-f) 분기 함수: validate_sql 의 다음 행선지를 정한다 ---
MAX_RETRIES = 3


def should_retry(state: AgentState) -> str:
    """반환값 'answer' / 'giveup' / 'retry' 중 하나는 아래 conditional_edges 의 키와 일치해야 한다."""
    if not state.get("error"):
        return "answer"   # 에러 없음 → 답변 생성으로
    if state.get("retry_count", 0) >= MAX_RETRIES:
        return "giveup"   # 너무 많이 시도 → 답변 노드에서 포기 메시지 출력
    return "retry"        # 다시 SQL 생성으로


# --- (5-g) 그래프 조립 ---
# 노드(함수) 4개를 등록하고, 엣지(연결)로 흐름을 그린다.
graph = StateGraph(AgentState)
graph.add_node("generate_sql", generate_sql)
graph.add_node("execute_sql", execute_sql)
graph.add_node("validate_sql", validate_sql)
graph.add_node("generate_answer", generate_answer)

graph.set_entry_point("generate_sql")              # 시작점
graph.add_edge("generate_sql", "execute_sql")      # 직선 연결
graph.add_edge("execute_sql", "validate_sql")
# 조건부 엣지: should_retry 의 반환값에 따라 다음 노드가 동적으로 결정됨.
graph.add_conditional_edges(
    "validate_sql",
    should_retry,
    {
        "answer": "generate_answer",
        "giveup": "generate_answer",
        "retry":  "generate_sql",   # 여기가 "재시도 루프" 의 핵심
    },
)
graph.add_edge("generate_answer", END)              # 종료

# .compile() 은 그래프를 실제 실행 가능한 객체로 만든다.
agent = graph.compile()
print("Agent compiled.")


# --- (5-h) 초기 상태 헬퍼 ---
# 모든 키를 빈 값으로 초기화한 dict 를 만들어 그래프 실행의 출발 상태로 쓴다.
def _initial_state(question: str) -> dict:
    return {
        "question": question,
        "sql": "",
        "result": [],
        "result_md": "",
        "answer": "",
        "error": "",
        "retry_count": 0,
    }

In [ ]:
# 그래프를 그림으로 그려서 "어떤 노드 → 어떤 노드" 흐름을 한눈에 본다.
# .draw_mermaid_png() 는 LangGraph 가 Mermaid 다이어그램을 PNG 이미지로 만들어 줍니다.
# Colab 에서는 보통 잘 뜨지만, 환경에 따라 실패할 수 있으니 폴백으로 mermaid 텍스트를 출력합니다.
from IPython.display import Image, display

try:
    display(Image(agent.get_graph().draw_mermaid_png()))
except Exception as e:
    # PNG 렌더에 실패해도 Mermaid 텍스트를 보여주면 충분히 그래프 구조를 이해할 수 있다.
    print(f"[info] mermaid png 렌더 실패 ({e}) → 텍스트로 출력:")
    print(agent.get_graph().draw_mermaid())

## 시연 — 세 가지 질문

아래 `ask_agent()` 헬퍼가 각 노드의 진행 상황을 스트리밍으로 출력합니다. **재시도 루프** 가 실제로 돌아가는 장면을 주목하세요 — 존재하지 않는 컬럼 힌트를 준 세 번째 질문에서는 `generate_sql` 이 두 번 이상 호출됩니다.

In [ ]:
def ask_agent(question: str) -> dict:
    """질문 하나를 에이전트에 흘려 보내고, 각 노드의 진행 상황을 사람이 보기 좋게 출력한다."""
    print(f"\n{'='*60}\n[Q] {question}\n{'='*60}")
    last = {}  # 마지막에 누적된 state 를 함수 외부로 돌려주기 위한 누산기
    # agent.stream(...) 은 노드 실행이 끝날 때마다 (노드명, 출력) 이벤트를 yield 한다.
    # 한 번에 끝까지 기다리지 않고 단계별로 출력하면 "재시도" 같은 흐름이 눈에 보인다.
    for event in agent.stream(_initial_state(question)):
        # event 는 {노드명: 그 노드가 반환한 dict} 형태. 보통 키가 한 개이지만 안전하게 순회.
        for node_name, output in event.items():
            last = {**last, **(output or {})}  # dict 병합 — Python 3.9+ 의 ** 언패킹 문법
            print(f"\n>> [{node_name}]")
            # 각 노드가 채운 키를 골라 한 줄씩 요약 출력
            if output.get("sql"):
                preview = output["sql"].replace("\n", " ")  # 한 줄로 짧게 보여주기
                print(f"   SQL : {preview[:120]}{'...' if len(preview) > 120 else ''}")
            if output.get("error"):
                print(f"   ERR : {output['error'][:120]}")
            if output.get("result_md"):
                first_line = output["result_md"].split("\n")[0][:80]
                print(f"   RES : {first_line} ...")
            if output.get("answer"):
                print(f"   ANS : {output['answer'][:200]}")
    return last


# 시연 1 — 단일 테이블 COUNT. 재시도 없이 한 번에 성공해야 한다.
_ = ask_agent("현재 등록된 환자 수는 몇 명인가요?")

In [ ]:
# 시연 2 — 여러 테이블 JOIN + GROUP BY 가 필요한 질문.
# LLM 이 스키마의 FOREIGN KEY 정보만으로 조인 조건을 스스로 추론한다는 점이 핵심.
_ = ask_agent("진료과별 의사 수를 많은 순서대로 보여주세요.")

In [ ]:
# 시연 3 — "재시도 루프" 가 실제로 도는 모습을 보기 위한 의도적 함정.
# 일부러 존재하지 않는 컬럼(is_active) 을 힌트로 줘서 첫 SQL 은 PostgreSQL 에러로 실패시킨다.
# 이때 validate_sql 분기에서 'retry' 가 반환 → generate_sql 이 에러 메시지를 보고 SQL 을 고친다.
trick_q = (
    "비활성 환자 목록을 is_active=false 컬럼 기준으로 조회해 주세요. "
    "(참고: 실제 스키마에 is_active 컬럼이 없을 가능성이 높습니다.)"
)
final = ask_agent(trick_q)
# 최종 state 를 보면 retry_count 가 2 이상이면 한 번이라도 재시도가 일어났다는 뜻.
print(f"\n[최종] retry_count = {final.get('retry_count', 0)}")
print(f"[최종] error       = '{final.get('error', '')[:120]}'")

## 방금 본 것을 해석하면

| 시연 | 관찰 포인트 |
|---|---|
| 1 | 단일 테이블 COUNT — 1번에 성공. `generate_sql → execute_sql → validate_sql → generate_answer` 직진. |
| 2 | JOIN + GROUP BY + ORDER BY — LLM이 스키마에서 FK 관계를 읽어 조인 조건을 스스로 채움. |
| 3 | 존재하지 않는 컬럼 때문에 첫 실행 실패 → `validate_sql` 분기 `retry` → `generate_sql` 재호출 → 에러 피드백을 프롬프트에 주입해 수정된 SQL 생성. 최대 3회 시도. |

## 4일 로드맵 — 이 에이전트를 쌓는 경로

| Day | 배우는 것 | 이 에이전트의 어느 부분? |
|---|---|---|
| **Day 1 (1~8H)** | PostgreSQL · SQL · LlamaIndex · Text-to-SQL 기초 + 프로젝트 브리핑 | DB 엔진 · SQL 문법 · 스키마 이해 |
| **Day 2 (9~12H)** | Text-to-SQL 심화 · 멀티턴 상담사 · Gradio UI | SQL 생성 프롬프트 · 가드레일 |
| **Day 3 (13~20H)** | Vanna · LangChain/LCEL · Advanced RAG · **LangGraph SQL 에이전트** | **오늘 본 4-노드 그래프를 직접 구현** |
| **Day 4 (21~24H)** | LangSmith 트레이싱 · Ragas 정량 평가 · 최종 발표 | 에이전트 관측 & 평가 |

## 환경 셋업 체크리스트

- [ ] Google Colab 접속 + 새 노트북
- [ ] Neon PostgreSQL 가입 & `?sslmode=require` 포함 DSN 복사
- [ ] Colab Secrets 등록: `OPENAI_API_KEY`, `NEON_DSN` (Notebook access: ON)
- [ ] 위 `%pip install` + 부트스트랩 셀이 에러 없이 완료
- [ ] 이 노트북의 시연 3개가 모두 "답변 문자열" 을 출력
- [ ] (선택) `01_postgres_basics.ipynb` 먼저 실행해 병원 DB DDL+시드 적재

> **4일 후 여러분은** 이 에이전트를 **본인 도메인 DB** 에 맞춰 직접 만들 수 있게 됩니다. 오늘은 이게 뭘 하는 건지 "구경" 하는 날입니다.

## 다음 노트북에서는...

`01_postgres_basics.ipynb` — 이 에이전트의 가장 기초 재료인 **PostgreSQL과 SQL**부터 쌓아 올립니다. Neon 가입, Colab 접속, 병원 샘플 DB 적재, `SELECT/WHERE/ORDER BY/LIMIT/EXPLAIN` 기본을 직접 실행합니다. 24시간 후 다시 이 00번 노트북으로 돌아오면 코드가 완전히 다르게 보일 것입니다.